Лабораторная работа 5.1. Методы оптимизации: методы многомерного поиска

In [4]:
%pip install gymnasium numpy gymnasium[toy-text]

Note: you may need to restart the kernel to use updated packages.


In [5]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
class PolicyIterationAgent:
    def __init__(self, env):
        self.env = env
        self.n_states = env.observation_space.n
        self.n_actions = env.action_space.n
        self.policy_probs = np.zeros((self.n_states, self.n_actions))
        self.policy_probs[:, 0] = 1.0 
        self.state_values = np.zeros(self.n_states)
        self.gamma = 0.95
        self.theta = 1e-8

    def train(self):
        print("Начало обучения...")
        for i in range(500):
            while True:
                delta = 0
                for s in range(self.n_states):
                    if s == 47: continue
                    v = self.state_values[s]
                    
                    new_v = 0
                    for a in range(self.n_actions):
                        prob_a = self.policy_probs[s, a]
                        for p, ns, r, done in self.env.unwrapped.P[s][a]:
                            future_v = self.state_values[ns] if not done else 0
                            new_v += prob_a * p * (r + self.gamma * future_v)
                    
                    self.state_values[s] = new_v
                    delta = max(delta, abs(v - new_v))
                if delta < self.theta: break
            
            policy_stable = True
            for s in range(self.n_states):
                if s == 47: continue
                old_action = np.argmax(self.policy_probs[s])
                
                q_sa = []
                for a in range(self.n_actions):
                    val = 0mmo
                    for p, ns, r, done in self.env.unwrapped.P[s][a]:
                        future_v = self.state_values[ns] if not done else 0
                        val += p * (r + self.gamma * future_v)
                    q_sa.append(val)
                
                best_action = np.argmax(q_sa)
                if old_action != best_action:
                    policy_stable = False
                
                self.policy_probs[s] = np.eye(self.n_actions)[best_action]
            
            if policy_stable:
                print(f"Обучение завершено. Стратегия стабилизировалась на итерации {i}")
                break

In [9]:
def create_animation(agent):
    env = gym.make('CliffWalking-v1', render_mode='rgb_array')
    state, _ = env.reset()
    frames = []
    path = []
    
    print("Запуск агента по выученной стратегии...")
    for step in range(50):
        frames.append(env.render())
        
        action = np.argmax(agent.policy_probs[state])
        path.append((state, action))
        
        state, reward, terminated, truncated, _ = env.step(action)

        print(path)

        if terminated or truncated:
            frames.append(env.render())
            print(f"Успех! Достигли цели за {step+1} шагов.")
            break
    
    if not (terminated or truncated):
        print("Агент не успел дойти за 50 шагов. Проверь логи пути ниже.")
    
    # Печатаем первые 10 шагов для диагностики
    print("Первые шаги (состояние, действие):", path[:10])
    
    env.close()

    # Анимация
    fig = plt.figure(figsize=(8, 3))
    plt.axis('off')
    img = plt.imshow(frames[0])

    def animate(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=250, blit=True)
    plt.close()
    return anim

In [10]:
env_setup = gym.make('CliffWalking-v1')
agent = PolicyIterationAgent(env_setup)
agent.train()

anim = create_animation(agent)
HTML(anim.to_jshtml())

Начало обучения...
Обучение завершено. Стратегия стабилизировалась на итерации 14
Запуск агента по выученной стратегии...
[(36, np.int64(0))]
[(36, np.int64(0)), (24, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1)), (27, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1)), (27, np.int64(1)), (28, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1)), (27, np.int64(1)), (28, np.int64(1)), (29, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1)), (27, np.int64(1)), (28, np.int64(1)), (29, np.int64(1)), (30, np.int64(1))]
[(36, np.int64(0)), (24, np.int64(1)), (25, np.int64(1)), (26, np.int64(1)), (27, np.int64(1)), (28, np.int64(1)), (29, np.int64(1)), (30, np.int64(1)), (31, np.int64(1))]
[(36, np.int64